<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Preprocess consistently

**[C05-M02-L03 · Preprocess consistently](https://learning.nextia-ai.com/courses/ml/m02/preprocess-consistently/)** · C05, Machine Learning: From Problem to Reliable Model · Module 2, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** combine one-hot encoding, imputation and scaling with a logistic regression in one scikit-learn pipeline, write it into `ticket_model.py`, prove that it learned only from the train set, and compare it with the baselines.

| | |
|---|---|
| **Time** | About 65 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Scores, `predict_proba` and `predict`, from [Classification intuition](https://learning.nextia-ai.com/courses/ml/m02/classification-intuition/). Fit on train only, from C04's [Fit transformations correctly](https://learning.nextia-ai.com/courses/sql/m05/fit-transformations-correctly/). |
| **You do not need** | The local project. The setup below downloads the data, and this notebook writes `ticket_model.py`. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m02/preprocess-consistently/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M02-L03-preprocess-consistently/preprocess-consistently-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the C05 practice data (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m01/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Four tickets, one ColumnTransformer

In Lesson 2, a text column and a column with gaps stopped the model. **Preprocessing** turns every feature into numbers with no gaps. Each step has a **fit** part, which learns numbers from data, and a **transform** part, which uses them. You met this in C04's [Fit transformations correctly](https://learning.nextia-ai.com/courses/sql/m05/fit-transformations-correctly/).

A `ColumnTransformer` sends each list of columns to its own steps:

- **One-hot encoding** (`OneHotEncoder`) turns one category column into one column of 0 and 1 for each category that fit saw.
- **Imputation** (`SimpleImputer`) fills a gap with a value learned in fit, here the median. `add_indicator=True` adds a column that is 1 where the value was missing.
- **Scaling** (`StandardScaler`) computes (value − mean) ÷ SD, with the mean and SD learned in fit.

> **Predict.** The known values of `order_value` are 20, 60 and 100. Which value fills the gap? Then run the cell.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

tiny = pd.DataFrame({"channel": ["email", "chat", "email", "phone"],
                     "order_value": [20.0, None, 60.0, 100.0]})
numbers = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                    ("scale", StandardScaler())])
prepare = ColumnTransformer([("categories", OneHotEncoder(handle_unknown="ignore"), ["channel"]),
                             ("numbers", numbers, ["order_value"])], verbose_feature_names_out=False)
prepare.fit(tiny)
print(pd.DataFrame(prepare.transform(tiny), columns=prepare.get_feature_names_out()).round(2).to_string())

> **Check.** You should see five columns: `channel_chat`, `channel_email`, `channel_phone`, `order_value` and `missingindicator_order_value`. The `order_value` column is −1.41, 0.0, 0.0, 1.41.

Check it by hand. The median of 20, 60 and 100 is 60, so the gap becomes 60. The values are then 20, 60, 60, 100: the mean is 60 and the SD is 28.28. So 20 becomes (20 − 60) ÷ 28.28 = −1.41. The scaler also scales the indicator column (0, 1, 0, 0 becomes −0.58, 1.73, −0.58, −0.58). That is harmless: it is still "missing or not".

Now a new ticket from a channel that fit never saw, `social`, with no order value. Run the cell.

In [ ]:
new_ticket = pd.DataFrame({"channel": ["social"], "order_value": [None]})
print(pd.DataFrame(prepare.transform(new_ticket), columns=prepare.get_feature_names_out()).round(2).to_string())

> **Check.** You should see 0.0 in the three channel columns, 0.0 for `order_value` and 1.73 for the indicator.

`handle_unknown="ignore"` gives an unknown channel 0 in every channel column. The gap gets the median learned in fit, 60, which scales to 0.0. Nothing is learned from the new ticket.

## 2. Write ticket_model.py

The real pipeline has the same shape, with all ten features. Priya puts it in a file, `ticket_model.py`, so that every lesson, script and notebook uses the same columns and the same steps. Run the cell. It writes the file in the project folder.

> **Warning.** If you change the file later, restart the kernel before you import it again. Python imports a file only once per session.

In [ ]:
%%writefile ticket_model.py
"""The shared parts of Larkfield's escalation model: the columns, how to load
a part of the data, and the preprocessing-plus-model pipeline.

Nextia Learning, C05, Module 2, lesson 3 (Preprocess consistently).
"""

from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA = Path("data")
CATEGORIES = ["channel", "team", "segment", "region"]
NUMBERS = ["priority", "order_value", "word_count", "customer_tenure_days", "prior_tickets_90d",
           "created_hour"]
FEATURES = CATEGORIES + NUMBERS
TARGET = "escalated_72h"


def read(path):
    """Read a ticket CSV file. Only an empty cell is missing: the text
    "unknown" stays a category."""
    return pd.read_csv(path, parse_dates=["created_at"], dtype={"customer_id": "string"},
                       keep_default_na=False, na_values=[""])


def load(part):
    """One part of the split: train, valid or test."""
    return read(DATA / f"{part}.csv")


def build_pipeline(model=None):
    """Preprocessing and model in one object. fit() learns the medians, the
    scaling and the categories from the rows it is given, and nothing else."""
    numbers = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    prepare = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORIES),
        ("numbers", numbers, NUMBERS),
    ])
    return Pipeline([("prepare", prepare), ("model", model if model is not None else LogisticRegression(max_iter=1000))])

> **Check.** You should see `Writing ticket_model.py` (or `Overwriting ticket_model.py` if you run it again).

`build_pipeline()` returns a **pipeline**: the preprocessing and the model in one object. `pipeline.fit(train)` fits every step on the train rows. `predict_proba(valid)` only transforms the validation rows, with what fit learned, and then scores them.

## 3. Fit on train, and look inside

Run the cell. It loads the train and validation sets and fits the pipeline on train. Then it prints the medians that the imputer learned, next to the medians of train and valid.

In [ ]:
from ticket_model import CATEGORIES, FEATURES, NUMBERS, TARGET, build_pipeline, load

train, valid = load("train"), load("valid")
pipeline = build_pipeline()
pipeline.fit(train[FEATURES], train[TARGET])

prepare = pipeline.named_steps["prepare"]
learned = prepare.named_transformers_["numbers"]["impute"].statistics_
print(pd.DataFrame({"learned": learned, "train median": train[NUMBERS].median(),
                    "valid median": valid[NUMBERS].median()}).to_string())

> **Check.** You should see the learned medians 3.0, 86.56, 89.0, 1178.0, 5.0 and 14.0. They are equal to the train medians in every row.

This is the proof that fit learned from the train set only. The learned medians equal the train medians, and four of them differ from the validation medians.

> **Your turn.** Which number columns would get a different median if fit had used the validation set? Put their names in `would_change`, as a sorted list. Compare `learned` with `valid[NUMBERS].median()`. Run the cell, then the check cell.

In [ ]:
would_change = ...  # a sorted list of column names
print(would_change)

In [ ]:
expect_hash('would_change', would_change, '5c3729840eca50b0')

Next, the categories that the encoder learned, and the number of columns that go into the model. Run the cell.

In [ ]:
encoder = prepare.named_transformers_["categories"]
for column, categories in zip(CATEGORIES, encoder.categories_):
    print(column, list(categories))
print(len(prepare.get_feature_names_out()), "columns go into the model")

> **Check.** You should see five channels, five teams, four segments and five regions, and `27 columns go into the model`.

27 = 19 one-hot columns + 6 numbers + 2 indicators. Only `order_value` and `customer_tenure_days` had gaps in train, so only they get an indicator.

Now score the validation set. Run the cell.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

scores = pipeline.predict_proba(valid[FEATURES])[:, 1]
print("ROC AUC on valid:", round(roc_auc_score(valid[TARGET], scores), 3))
print("AP on valid:", round(average_precision_score(valid[TARGET], scores), 3))

> **Check.** You should see `ROC AUC on valid: 0.738` and `AP on valid: 0.353`.

## 4. Results: baseline, rule and model

Every result goes next to the baselines. The next cell defines `summary`: it prints the flags, precision and recall of a decision, and the ROC AUC and AP of a score. Run it. It prints nothing.

In [ ]:
def summary(name, scores, flags):
    """One line of the results table, on the validation set."""
    flagged = int(flags.sum())
    caught = int((flags & (valid[TARGET] == 1)).sum())
    precision = caught / flagged if flagged else float("nan")
    recall = caught / valid[TARGET].sum()
    auc = roc_auc_score(valid[TARGET], scores)
    ap = average_precision_score(valid[TARGET], scores)
    print(f"{name:9s} flagged {flagged:3d} ({flagged / len(valid):.1%}), precision {precision:.3f}, "
          f"recall {recall:.3f}, ROC AUC {auc:.3f}, AP {ap:.3f}")

The majority baseline goes through the same pipeline, with `DummyClassifier` as the model: it gives every ticket the train rate as its score. The rule's 0/1 flags are also its score.

> **Predict.** Which of the three has the highest ROC AUC? Which flags the most tickets? Then run the cell.

In [ ]:
from sklearn.dummy import DummyClassifier

majority = build_pipeline(DummyClassifier(strategy="prior")).fit(train[FEATURES], train[TARGET])
summary("majority", majority.predict_proba(valid[FEATURES])[:, 1], majority.predict(valid[FEATURES]))

rule = (valid["priority"] == 1).astype(int)
summary("rule", rule, rule)

summary("logistic", scores, pipeline.predict(valid[FEATURES]))

> **Check.** You should see:
>
> - `majority  flagged   0 (0.0%), precision nan, recall 0.000, ROC AUC 0.500, AP 0.155`
> - `rule      flagged 163 (13.8%), precision 0.374, recall 0.333, ROC AUC 0.615, AP 0.228`
> - `logistic  flagged  18 (1.5%), precision 0.556, recall 0.055, ROC AUC 0.738, AP 0.353`

The model ranks much better than the rule (AP 0.353 against 0.228). But `predict` uses 0.5 and flags only 18 tickets. Compare decisions at the same workload: that is the module practice below.

## 5. Change one thing: remove the scaler

`set_params` changes one step without editing the file. `"passthrough"` means "skip this step".

> **Predict.** Without the scaler, will the score change a lot? Will anything else happen? Then run the cell.

In [ ]:
import warnings

unscaled = build_pipeline().set_params(prepare__numbers__scale="passthrough")
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")
    unscaled.fit(train[FEATURES], train[TARGET])
for w in caught_warnings:
    print(type(w.message).__name__ + ":", str(w.message).splitlines()[0])
unscaled_scores = unscaled.predict_proba(valid[FEATURES])[:, 1]
print("ROC AUC", round(roc_auc_score(valid[TARGET], unscaled_scores), 3),
      "AP", round(average_precision_score(valid[TARGET], unscaled_scores), 3))

> **Check.** You should see `ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):`, then `ROC AUC 0.739 AP 0.354`.

The scores hardly change. But the solver did not finish. `order_value` (up to thousands) and `customer_tenure_days` (up to about 2,000) have a much larger scale than the 0/1 columns. The fitted weights are not the best ones, and their sizes cannot be compared. Keep the scaler.

## 6. Failure case: preprocessing by hand

Tomás prepares the features with pandas: `get_dummies` for the categories and `fillna` for the gaps. It works on the whole validation set. Then the routing service sends one new ticket.

> **Predict.** What happens with one ticket? Then run the cell.

In [ ]:
from sklearn.linear_model import LogisticRegression

by_hand = pd.get_dummies(train[FEATURES], columns=CATEGORIES, dtype=int).fillna(train[NUMBERS].median())
with warnings.catch_warnings(action="ignore"):  # no scaler: hide the ConvergenceWarning
    model = LogisticRegression(max_iter=1000).fit(by_hand, train[TARGET])
one_ticket = pd.get_dummies(valid[FEATURES].head(1), columns=CATEGORIES, dtype=int)
print(len(by_hand.columns), "columns in train,", len(one_ticket.columns), "for one ticket")
try:
    model.predict_proba(one_ticket)
except ValueError as error:
    print("ValueError:", str(error).splitlines()[0])

> **Check.** You should see `25 columns in train, 10 for one ticket`, then `ValueError: The feature names should match those that were passed during fit.`

**Cause:** `get_dummies` learns nothing. It makes one column for each category in the rows it gets: one ticket has one channel, so it gets one channel column. **Fix:** an encoder that is fitted on train and only transforms later rows, inside the pipeline. `build_pipeline()` gives the same 27 columns for one ticket or for a thousand.

## 7. Module practice: the model against the rule, at the same workload

The rule flags 163 valid tickets. Is the model better if it also flags 163: the 163 tickets with the highest scores?

> **Your turn.** Flag the 163 valid tickets with the highest `scores`. Count how many were escalated (`caught_163`). Compute the precision and the recall, rounded to 3 decimals. Hint: `scores.argsort()[::-1][:163]` gives the positions of the 163 highest scores. Run the cell, then the check cell. The [lesson page](https://learning.nextia-ai.com/courses/ml/m02/preprocess-consistently/#module-practice) has the model answer.

In [ ]:
top = scores.argsort()[::-1][:163]
caught_163 = ...
precision_163 = ...
recall_163 = ...
print(caught_163, precision_163, recall_163)

In [ ]:
expect_hash('[caught_163, precision_163, recall_163]', [caught_163, precision_163, recall_163], '0bf8472a5fa222a5')

## 8. Module check: reproduce and confirm

The completion check of the module has two parts. Do them here, and write the answer on the [lesson page](https://learning.nextia-ai.com/courses/ml/m02/preprocess-consistently/#module-check).

1. **Reproduce.** Choose **Restart and run all** (in Colab: **Runtime › Restart session and run all**). Section 3 must print `ROC AUC on valid: 0.738` and `AP on valid: 0.353`, and section 4 the three lines of the results table.
2. **Confirm.** Run the next cell. It checks that every learned median equals the train median. It also checks that the scaler's means equal the means of the filled train columns.

In [ ]:
import numpy as np

numbers_step = prepare.named_transformers_["numbers"]
filled = numbers_step["impute"].transform(train[NUMBERS])
print("medians equal train medians:", np.allclose(numbers_step["impute"].statistics_, train[NUMBERS].median()))
print("means equal train means:", np.allclose(numbers_step["scale"].mean_, filled.mean(axis=0)))
print("medians equal valid medians:", np.allclose(numbers_step["impute"].statistics_, valid[NUMBERS].median()))

> **Check.** You should see `True`, `True` and `False`. The pipeline learned from the train set, and not from the validation set.

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m02/preprocess-consistently/#module-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you built one pipeline for all ten features: one-hot encoding, median imputation with indicators, scaling and a logistic regression. You wrote it into `ticket_model.py`. Fit learned the medians and categories from train only: the medians equal train's. On valid, the model reaches ROC AUC 0.738 and AP 0.353, against 0.615 and 0.228 for the rule. At the rule's workload of 163 flags, it catches 66 escalations instead of 61. At 0.5, it flags only 18 tickets.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Trees and ensembles](https://learning.nextia-ai.com/courses/ml/m03/trees-and-ensembles/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m02/preprocess-consistently/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.